# XGBoost Fraud Detection with Balanced Sampling

This notebook trains an XGBoost model on fraud detection data with:
- All fraud cases (fraud_flag = 1)
- 10% sample of non-fraud cases (fraud_flag = 0)

**Date Range:** June 2025 (1 month)

## 1. Setup and Imports

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml import Pipeline

import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_style('whitegrid')

print("✅ Imports successful")
print(f"XGBoost version: {xgb.__version__}")

## 2. Initialize Spark Session

In [ ]:
# Stop any existing sessions
try:
    existing = SparkSession.getActiveSession()
    if existing:
        existing.stop()
        print("Stopped existing Spark session")
except:
    pass

# Package dependencies
packages = [
    "com.clickhouse.spark:clickhouse-spark-runtime-3.5_2.12:0.8.1",
    "com.clickhouse:clickhouse-client:0.9.4",
    "com.clickhouse:clickhouse-http-client:0.9.4",
    "org.apache.httpcomponents.client5:httpclient5:5.2.1"
]

# Create Spark session
spark = (SparkSession.builder
    .appName("XGBoost-Fraud-Detection")
    .master("spark://10.205.161.118:7077")
    .config("spark.jars.packages", ",".join(packages))
    .config("spark.executor.memory", "150g")
    .config("spark.executor.memoryOverhead", "5g")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.cores", "32")
    .config("spark.executor.instances", "2")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.default.parallelism", "96")
    .getOrCreate()
)

# Configure ClickHouse catalog
spark.conf.set("spark.sql.catalog.clickhouse", "com.clickhouse.spark.ClickHouseCatalog")
spark.conf.set("spark.sql.catalog.clickhouse.host", "localhost")
spark.conf.set("spark.sql.catalog.clickhouse.protocol", "http")
spark.conf.set("spark.sql.catalog.clickhouse.http_port", "8123")
spark.conf.set("spark.sql.catalog.clickhouse.user", "default")
spark.conf.set("spark.sql.catalog.clickhouse.password", "DfsTeChB1")
spark.conf.set("spark.sql.catalog.clickhouse.database", "public")
spark.conf.set("spark.clickhouse.write.format", "json")

print("✅ Spark session initialized")
print(f"   Version: {spark.version}")
print(f"   Master: {spark.sparkContext.master}")

## 3. Load Data with Sampling Strategy

Loading:
- **All fraud cases** (fraud_flag = 1)
- **10% sample** of non-fraud cases (fraud_flag = 0)

In [ ]:
# Configuration
START_DATE = '2025-06-01'
END_DATE = '2025-06-30'
NON_FRAUD_SAMPLE_RATE = 0.10
RANDOM_SEED = 42

# Feature columns
FEATURE_COLS = [
    'cutoff_date', 'fraud_flag', 'trx_channel', 'trx_type', 
    'start_balance', 'trx_amt', 'mbar_registered_channel',
    'hour_of_day', 'day_of_week', 'is_weekend', 'is_night', 
    'is_business_hours', 'is_unusual_hour', 'night_weekend_combo',
    'txn_txns_3d', 'txn_total_amount_3d', 'txn_avg_amount_3d',
    'txn_max_amount_3d', 'txn_min_amount_3d', 'txn_unique_recipients_3d',
    'txn_unique_channels_3d', 'txn_unique_types_3d', 'txn_is_high_activity_3d',
    'txn_multi_channel_recent', 'txn_amount_deviation_from_avg',
    'txn_night_txns_3d', 'txn_weekend_txns_3d',
    'channel_new_jc_app', 'channel_ussd', 'channel_ussd_api',
    'channel_payment_gateway', 'channel_mobile_app',
    'type_transfer_c2c', 'type_transfer_c2b', 'type_bill_payment',
    'type_mobile_load', 'user_total_txns_3d', 'user_total_amount_3d',
    'user_avg_amount_3d', 'user_max_amount_3d', 'user_unique_recipients_3d',
    'user_unique_channels_3d', 'user_total_txns_7d', 'user_avg_amount_7d',
    'user_max_amount_7d', 'user_night_txns_7d', 'user_weekend_txns_7d'
]

print(f"📅 Date Range: {START_DATE} to {END_DATE}")
print(f"📊 Features: {len(FEATURE_COLS)} columns")

In [ ]:
# Load ALL fraud cases
print("\n" + "="*80)
print("LOADING FRAUD CASES (fraud_flag = 1)")
print("="*80)

fraud_query = f"""
    SELECT {', '.join(FEATURE_COLS)}
    FROM clickhouse.public.stixor_fraud_features_distributed
    WHERE cutoff_date BETWEEN '{START_DATE}' AND '{END_DATE}'
        AND mbar_account_type_name = 'Customer Account'
        AND fraud_flag = 1
"""

start_time = time.time()
df_fraud = spark.sql(fraud_query)
fraud_count = df_fraud.count()
fraud_duration = time.time() - start_time

print(f"✅ Loaded {fraud_count:,} fraud cases in {fraud_duration:.2f}s")

In [ ]:
# Load and sample NON-FRAUD cases (10%)
print("\n" + "="*80)
print(f"LOADING NON-FRAUD CASES (fraud_flag = 0, {NON_FRAUD_SAMPLE_RATE*100:.0f}% sample)")
print("="*80)

non_fraud_query = f"""
    SELECT {', '.join(FEATURE_COLS)}
    FROM clickhouse.public.stixor_fraud_features_distributed
    WHERE cutoff_date BETWEEN '{START_DATE}' AND '{END_DATE}'
        AND mbar_account_type_name = 'Customer Account'
        AND fraud_flag = 0
"""

start_time = time.time()
df_non_fraud_full = spark.sql(non_fraud_query)
non_fraud_full_count = df_non_fraud_full.count()

# Sample 10%
df_non_fraud = df_non_fraud_full.sample(
    withReplacement=False, 
    fraction=NON_FRAUD_SAMPLE_RATE, 
    seed=RANDOM_SEED
)
non_fraud_count = df_non_fraud.count()
non_fraud_duration = time.time() - start_time

print(f"📊 Total non-fraud cases available: {non_fraud_full_count:,}")
print(f"✅ Sampled {non_fraud_count:,} non-fraud cases ({NON_FRAUD_SAMPLE_RATE*100:.0f}%) in {non_fraud_duration:.2f}s")

In [ ]:
# Combine datasets
print("\n" + "="*80)
print("COMBINING DATASETS")
print("="*80)

df_combined = df_fraud.union(df_non_fraud)
total_count = df_combined.count()

print(f"✅ Combined dataset: {total_count:,} rows")
print(f"\n📊 Class Distribution:")
print(f"   Fraud (1):     {fraud_count:,} ({fraud_count/total_count*100:.2f}%)")
print(f"   Non-Fraud (0): {non_fraud_count:,} ({non_fraud_count/total_count*100:.2f}%)")
print(f"   Class Ratio:   1:{non_fraud_count/fraud_count:.2f}")

## 4. Data Preprocessing

In [ ]:
# Identify categorical and numeric columns
CATEGORICAL_COLS = ['trx_channel', 'trx_type', 'mbar_registered_channel']
EXCLUDE_COLS = ['fraud_flag', 'cutoff_date', 'mbar_account_type_name']
NUMERIC_COLS = [f for f in FEATURE_COLS if f not in CATEGORICAL_COLS + EXCLUDE_COLS]

print(f"📊 Feature Types:")
print(f"   Categorical: {len(CATEGORICAL_COLS)} features")
print(f"   Numeric: {len(NUMERIC_COLS)} features")
print(f"   Target: fraud_flag")

In [ ]:
# Build preprocessing pipeline
print("\n" + "="*80)
print("BUILDING PREPROCESSING PIPELINE")
print("="*80)

stages = []
indexed_cols = []

# String indexing for categorical features
for col_name in CATEGORICAL_COLS:
    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=f"{col_name}_idx",
        handleInvalid="keep"
    )
    stages.append(indexer)
    indexed_cols.append(f"{col_name}_idx")

# Combine all feature columns
all_feature_cols = NUMERIC_COLS + indexed_cols

# Vector assembler
assembler = VectorAssembler(
    inputCols=all_feature_cols,
    outputCol="features",
    handleInvalid="skip"
)
stages.append(assembler)

# Create and fit pipeline
pipeline = Pipeline(stages=stages)

print("Fitting preprocessing pipeline...")
start_time = time.time()
pipeline_model = pipeline.fit(df_combined)
duration = time.time() - start_time

print(f"✅ Pipeline fitted in {duration:.2f}s")
print(f"   Total features: {len(all_feature_cols)}")

In [ ]:
# Transform data
print("\nTransforming data...")
start_time = time.time()
df_transformed = pipeline_model.transform(df_combined)
duration = time.time() - start_time

print(f"✅ Data transformed in {duration:.2f}s")

## 5. Split Data for Training and Testing

In [ ]:
# Train-test split (80-20)
print("\n" + "="*80)
print("SPLITTING DATA")
print("="*80)

train_df, test_df = df_transformed.randomSplit([0.8, 0.2], seed=RANDOM_SEED)

train_count = train_df.count()
test_count = test_df.count()

print(f"✅ Train set: {train_count:,} rows ({train_count/total_count*100:.1f}%)")
print(f"✅ Test set:  {test_count:,} rows ({test_count/total_count*100:.1f}%)")

# Check class distribution in splits
train_fraud = train_df.filter(col('fraud_flag') == 1).count()
test_fraud = test_df.filter(col('fraud_flag') == 1).count()

print(f"\n📊 Class Distribution:")
print(f"   Train - Fraud: {train_fraud:,}, Non-Fraud: {train_count-train_fraud:,}")
print(f"   Test  - Fraud: {test_fraud:,}, Non-Fraud: {test_count-test_fraud:,}")

In [ ]:
# Convert to Pandas for XGBoost
print("\nConverting to Pandas DataFrames...")
start_time = time.time()

# Select only needed columns
train_pd = train_df.select('features', 'fraud_flag').toPandas()
test_pd = test_df.select('features', 'fraud_flag').toPandas()

duration = time.time() - start_time
print(f"✅ Converted in {duration:.2f}s")

# Extract features from SparseVector
def extract_features(df):
    features = np.array([x.toArray() for x in df['features']])
    labels = df['fraud_flag'].values
    return features, labels

X_train, y_train = extract_features(train_pd)
X_test, y_test = extract_features(test_pd)

print(f"\n✅ Feature matrices created:")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_test shape:  {X_test.shape}")

## 6. Train XGBoost Model

In [ ]:
print("\n" + "="*80)
print("TRAINING XGBOOST MODEL")
print("="*80)

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\n⚖️  Class imbalance ratio: {scale_pos_weight:.2f}")
print(f"   (Using scale_pos_weight to handle imbalance)")

# XGBoost parameters
params = {
    'objective': 'binary:logistic',
    'eval_metric': ['auc', 'logloss'],
    'max_depth': 10,
    'learning_rate': 0.1,
    'n_estimators': 200,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'random_state': RANDOM_SEED,
    'tree_method': 'hist',
    'n_jobs': -1
}

print(f"\n🔧 XGBoost Parameters:")
for key, value in params.items():
    print(f"   {key}: {value}")

In [ ]:
# Train model
print("\nTraining XGBoost model...")
start_time = time.time()

model = xgb.XGBClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=20
)

training_duration = time.time() - start_time
print(f"\n✅ Model trained in {training_duration:.2f}s ({training_duration/60:.2f} min)")

## 7. Model Evaluation

In [ ]:
print("\n" + "="*80)
print("MODEL EVALUATION")
print("="*80)

# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n📊 Performance Metrics:")
print(f"   Accuracy:  {accuracy:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {f1:.4f}")
print(f"   AUC-ROC:   {auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\n📊 Confusion Matrix:")
print(f"   TN: {cm[0,0]:,}  |  FP: {cm[0,1]:,}")
print(f"   FN: {cm[1,0]:,}  |  TP: {cm[1,1]:,}")

# Classification Report
print(f"\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-Fraud', 'Fraud']))

## 8. Visualizations

In [ ]:
# Confusion Matrix Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non-Fraud', 'Fraud'],
            yticklabels=['Non-Fraud', 'Fraud'],
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - XGBoost Model', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print("✅ Confusion matrix plotted")

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fpr, tpr, linewidth=2, label=f'XGBoost (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve - XGBoost Model', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ ROC curve plotted")

In [ ]:
# Precision-Recall Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(recall_vals, precision_vals, linewidth=2, label='XGBoost')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve - XGBoost Model', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Precision-Recall curve plotted")

## 9. Feature Importance Analysis

In [ ]:
# Extract feature importance
feature_importance = model.feature_importances_

# Create DataFrame
importance_df = pd.DataFrame({
    'feature': all_feature_cols,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("\n" + "="*80)
print("FEATURE IMPORTANCE (Top 20)")
print("="*80)
print(importance_df.head(20).to_string(index=False))

# Save to CSV
csv_path = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/xgboost_feature_importance.csv'
importance_df.to_csv(csv_path, index=False)
print(f"\n💾 Feature importance saved to: {csv_path}")

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(12, 10))
top_n = 25
top_features = importance_df.head(top_n)

ax.barh(range(len(top_features)), top_features['importance'].values, color='steelblue')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'].values)
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title(f'Top {top_n} Feature Importance - XGBoost Model', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Feature importance plotted")

## 10. Save Model

In [ ]:
# Save XGBoost model
model_path = '/root/research-dir/dev/jazzcash-fraud-detection/models/xgboost_fraud_model.json'
model.save_model(model_path)
print(f"✅ XGBoost model saved to: {model_path}")

# Save preprocessing pipeline
pipeline_path = '/root/research-dir/dev/jazzcash-fraud-detection/models/xgboost_preprocessing_pipeline'
pipeline_model.write().overwrite().save(pipeline_path)
print(f"✅ Preprocessing pipeline saved to: {pipeline_path}")

# Save metrics
metrics = {
    'model': 'XGBoost',
    'date_range': f'{START_DATE} to {END_DATE}',
    'training_samples': int(train_count),
    'test_samples': int(test_count),
    'fraud_samples': int(fraud_count),
    'non_fraud_samples': int(non_fraud_count),
    'sample_rate': NON_FRAUD_SAMPLE_RATE,
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'auc_roc': float(auc),
    'training_time_seconds': float(training_duration)
}

metrics_path = '/root/research-dir/dev/jazzcash-fraud-detection/analysis/xgboost_metrics.json'
import json
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"✅ Metrics saved to: {metrics_path}")

## 11. Summary

In [ ]:
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)
print(f"\n📅 Date Range: {START_DATE} to {END_DATE}")
print(f"\n📊 Data:")
print(f"   Total samples: {total_count:,}")
print(f"   - Fraud: {fraud_count:,} (100%)")
print(f"   - Non-Fraud: {non_fraud_count:,} ({NON_FRAUD_SAMPLE_RATE*100:.0f}% of available)")
print(f"   Train/Test split: 80/20")
print(f"\n🔧 Model: XGBoost Classifier")
print(f"   Features: {len(all_feature_cols)}")
print(f"   Training time: {training_duration:.2f}s")
print(f"\n📊 Performance:")
print(f"   AUC-ROC: {auc:.4f}")
print(f"   Accuracy: {accuracy:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall: {recall:.4f}")
print(f"   F1-Score: {f1:.4f}")
print(f"\n💾 Saved:")
print(f"   - Model: {model_path}")
print(f"   - Pipeline: {pipeline_path}")
print(f"   - Feature importance: {csv_path}")
print(f"   - Metrics: {metrics_path}")
print("\n" + "="*80)
print("✅ TRAINING COMPLETE")
print("="*80)

In [ ]:
# Stop Spark session
spark.stop()
print("\n✅ Spark session stopped")